In [ ]:
! pip install langchain-community

In [ ]:
! pip install tqdm

In [ ]:
! pip install chromadb

In [ ]:
! pip install python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

In [ ]:
load_dotenv(find_dotenv(), override=False)

In [ ]:
import os

GROQ_API_KEY= os.getenv("GROQ_API_KEY")
HF_TOKEN= os.getenv("HF_TOKEN")
CHROMA_API_KEY = os.getenv("CHROMA_API_KEY")
LANG_SMITH_KEY = os.getenv("LANG_SMITH_KEY")

In [ ]:
from chromadb import Schema, SparseVectorIndexConfig, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction

schema = Schema()

In [ ]:
import chromadb

client = chromadb.CloudClient(
  api_key=CHROMA_API_KEY,
  tenant="f6b39b05-f3b9-4614-896c-97cc6be82f15",
  database='hybrid_rag_alzheimers'
)

client.list_collections()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

In [ ]:
loader = DirectoryLoader('data/medical', glob="*.txt", show_progress=True, loader_cls=TextLoader)

In [ ]:
kb_docs = loader.load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=[". ", "? ", "! ", "\n\n", "\n", ", ", " "],
    keep_separator='end',
    is_separator_regex=False,
    chunk_size=500,
    chunk_overlap=75,
    length_function=len
)


In [ ]:
chunks = text_splitter.split_documents(kb_docs)

In [ ]:
print("Number of Documents:", len(kb_docs))
print()
print("Number of Chunks:", len(chunks))

In [ ]:
from chromadb import Schema, SparseVectorIndexConfig, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction

schema = Schema()

In [ ]:
schema = schema.create_index(
  key='sparse_vector_key',    
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)

In [ ]:
! pip install  langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
model_name = "BAAI/bge-base-en-v1.5"

In [ ]:
model_kwargs = {'device': 'cpu'}

In [ ]:
encode_kwargs = {'normalize_embeddings': True}

In [ ]:
print(HF_TOKEN)

In [ ]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
! pip install sentence-transformers

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

In [ ]:
from chromadb import Schema, VectorIndexConfig

In [ ]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

In [ ]:
import chromadb.utils.embedding_functions as embedding_functions

In [ ]:
from typing import List, Sequence
from langchain_community.embeddings import HuggingFaceEmbeddings
from chromadb.utils.embedding_functions import EmbeddingFunction
import chromadb

In [ ]:
hf_embedder  = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

In [ ]:

class HFLangChainEmbeddingFunction(EmbeddingFunction):
    def __init__(self, hf: HuggingFaceEmbeddings, use_query_head: bool = False):
        self.hf = hf
        self.use_query_head = use_query_head

    def __call__(self, input: Sequence[str]) -> List[List[float]]:
        # Chroma will pass a list of strings
        if self.use_query_head:
            return self.hf.embed_query(input)  # Some LC versions accept List[str]; if not, map()
        # Fallback to document head
        return self.hf.embed_documents(list(input))


In [ ]:
embedding_fn = HFLangChainEmbeddingFunction(hf_embedder, use_query_head=False)

In [ ]:
schema = schema.create_index(
  key='sparse_vector_key',    
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)

In [ ]:
schema = schema.create_index(
    config=VectorIndexConfig(
        space="cosine",
        embedding_function=embedding_fn
    )
)

In [ ]:
collection = client.get_or_create_collection(
  "medical_hybrid_rag_alzheimers",
  schema=schema, 
)

collection.count()

In [ ]:
collection.peek()

In [ ]:
print("Number of Documents:", len(kb_docs))
print()
print("Number of Chunks:", len(chunks))

In [ ]:
for chunk in chunks:
    print(chunk)

In [ ]:
! pip install snowballstemmer

In [ ]:
import uuid

collection.add(
    documents = [chunk.page_content for chunk in chunks],
    ids=[ str(uuid.uuid4()) for i in range(len(chunks)) ],
    metadatas= [chunk.metadata for chunk in chunks]
)

In [ ]:
from chromadb import Knn
from chromadb import Search

In [ ]:
sparse_rank = Knn(
  query="psychosis due to dementia",  # Text query for sparse embeddings
  key="sparse_vector_key",  # Metadata field for sparse vectors
  return_rank=True,
  limit=5                 # Only the 5 nearest documents get scored (default limit 16)
)

In [ ]:
sparse_search = (Search()
  .rank(sparse_rank)
  .select(K.DOCUMENT, K.SCORE)
  .limit(3)
)

In [ ]:
sparse_results = collection.search(sparse_search)

print(sparse_results)

In [ ]:
dense_rank = Knn(
  query="psychosis due to dementia",  # Text query for dense embeddings
  key="#embedding",          # Default embedding field
  return_rank=True,
)

In [ ]:
dense_search = (Search()
  .rank(dense_rank)
  .select(K.DOCUMENT, K.SCORE, K.METADATA)
  .limit(4)
)
  
dense_results = collection.search(dense_search)

print(dense_results)

In [ ]:
! pip install langchain-groq

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
print(GROQ_API_KEY)

In [211]:
chat_model = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=None,           
    reasoning_format="hidden", 
    max_retries=2,
    model_kwargs={      
        "top_p": 1.0,
      },
)

In [ ]:
query = input("Enter your question: ")


In [212]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """ROLE: Multi-Query Expansion Generator for Hybrid Retrieval (Semantic + BM25) in MEDICAL (Alzheimer’s disease)

TASK
Rewrite the user’s question into EXACTLY 5 distinct queries optimized for both dense semantic search and sparse BM25. Output JSON with a "queries" array of 5 strings and an optional short "rationale". No extra text.

GUIDELINES
- Domain scope: Alzheimer’s disease and related disorders (AD, prodromal AD, MCI due to AD, dementia differentials).
- Coverage: Include synonyms, abbreviations, and clinically relevant variants (e.g., “Alzheimer’s”, “AD”, “probable AD”, “MCI”, “amnestic MCI”, “dementia”, “neurocognitive disorder”).
- Biomarkers/diagnostics: Consider amyloid/tau (Aβ, p‑tau, CSF, PET), neurodegeneration (MRI hippocampal atrophy, FDG-PET), AT(N) framework, genetic risk (APOE ε4).
- Therapies: Include cholinesterase inhibitors (donepezil, rivastigmine, galantamine), memantine, disease-modifying anti-amyloid (e.g., lecanemab, donanemab), non‑pharmacologic interventions, lifestyle, caregiver support.
- Staging/assessment: MoCA, MMSE, CDR, functional decline, ADLs/IADLs, behavioral & psychological symptoms (BPSD), sleep, agitation.
- BM25-friendly: Ensure some variants contain exact medical tokens, drug names, biomarker terms, test names, and likely keyword pairings; include AND-like co-occurrence phrasing.
- Semantic-friendly: Include natural, conversational paraphrases that capture intent, context, and outcomes.
- Evidence/time filters (use if implied): guideline terms (e.g., “guideline”, “criteria”, “AAN”, “NIA-AA”), dates/time scopes (“recent”, “since 2022”).
- Safety: Do NOT invent facts, diagnoses, or treatment recommendations. Avoid speculative claims. Stay within the query’s scope.
- Distinctness: No trivial reorderings—each query should add different token mixes or intent angles.
- Quantity: ALWAYS output exactly 5 queries. If fewer are meaningful, include best variants that differ in phrasing and keyword composition.

STRICT OUTPUT FORMAT(DO NOT DEVIATE THE OUTPUT FORMAT)
[query1,query2,query3,query4,query5]

USER QUERY
{user_query}
"""

In [213]:
print(PROMPT_TEMPLATE)

ROLE: Multi-Query Expansion Generator for Hybrid Retrieval (Semantic + BM25) in MEDICAL (Alzheimer’s disease)

TASK
Rewrite the user’s question into EXACTLY 5 distinct queries optimized for both dense semantic search and sparse BM25. Output JSON with a "queries" array of 5 strings and an optional short "rationale". No extra text.

GUIDELINES
- Domain scope: Alzheimer’s disease and related disorders (AD, prodromal AD, MCI due to AD, dementia differentials).
- Coverage: Include synonyms, abbreviations, and clinically relevant variants (e.g., “Alzheimer’s”, “AD”, “probable AD”, “MCI”, “amnestic MCI”, “dementia”, “neurocognitive disorder”).
- Biomarkers/diagnostics: Consider amyloid/tau (Aβ, p‑tau, CSF, PET), neurodegeneration (MRI hippocampal atrophy, FDG-PET), AT(N) framework, genetic risk (APOE ε4).
- Therapies: Include cholinesterase inhibitors (donepezil, rivastigmine, galantamine), memantine, disease-modifying anti-amyloid (e.g., lecanemab, donanemab), non‑pharmacologic interventions

In [214]:
prompt_template = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE
    ]
)

In [ ]:
! pip install --upgrade --quiet langchain-groq

In [216]:
from langchain_groq import ChatGroq

In [217]:
query_chat_model = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=1,
    max_tokens=None,           
    reasoning_format="hidden", 
    max_retries=2,
)

In [ ]:
# from langchain_core.output_parsers import JsonOutputParser

In [218]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# query_parser = JsonOutputParser()

In [219]:
query_parser = StrOutputParser()

In [220]:
query_chain = prompt_template | query_chat_model | query_parser

In [ ]:
from langchain_core.runnables import RunnablePassthrough

In [ ]:
query = input("Enter your question: ")
generator_chain.invoke({"user_query":query})

In [ ]:
query = input("Enter your question: ")
generator_chain.invoke({"user_query":query})

In [ ]:
print(query_output)

In [ ]:

import json
import re
import ast
from typing import List, Tuple, Optional, Union


In [ ]:

def _strip_fences(text: str) -> str:
    # Remove ```...``` fences if present
    m = re.search(r"```(?:json|python)?\s*([\s\S]*?)```", text, flags=re.I)
    return m.group(1).strip() if m else text.strip()


In [ ]:

def _loads_lenient(text: str) -> Union[dict, list]:
    """
    Try JSON first; if it fails (e.g., single quotes / Python dict),
    fall back to ast.literal_eval safely.
    """
    try:
        return json.loads(text)
    except Exception:
        # Attempt to extract the first JSON-ish object/list if surrounding prose exists
        # Try object
        m = re.search(r"(\{[\s\S]*\})", text)
        if m:
            chunk = m.group(1)
            # Try JSON again
            try:
                return json.loads(chunk)
            except Exception:
                # Try Python literal
                try:
                    return ast.literal_eval(chunk)
                except Exception:
                    pass
        # Try list
        m = re.search(r"(\[[\s\S]*\])", text)
        if m:
            chunk = m.group(1)
            try:
                return json.loads(chunk)
            except Exception:
                try:
                    return ast.literal_eval(chunk)
                except Exception:
                    pass
        # Final fallback: Python literal on whole text
        return ast.literal_eval(text)

def parse_llm_multiquery(raw_text: str) -> Tuple[List[str], Optional[str]]:
    """
    Accepts:
      - JSON array: ["q1","q2",...]
      - JSON object: {"queries":[...], "rationale":"..."}
      - Python-style dict/list with single quotes
      - With or without ```json fences
    Returns: (queries, rationale)
    """
    cleaned = _strip_fences(raw_text)
    data = _loads_lenient(cleaned)

    # Normalize shapes
    queries: List[str]
    rationale: Optional[str] = None

    if isinstance(data, list):
        queries = [str(x).strip() for x in data if isinstance(x, str)]
    elif isinstance(data, dict):
        # keys may not exist; be defensive
        q = data.get("queries")
        if isinstance(q, list):
            queries = [str(x).strip() for x in q if isinstance(x, str)]
        else:
            # If dict but no 'queries', try to coerce any list-like value
            list_like = next((v for v in data.values() if isinstance(v, list)), [])
            queries = [str(x).strip() for x in list_like if isinstance(x, str)]
        rationale_val = data.get("rationale")
        if isinstance(rationale_val, str):
            rationale = rationale_val.strip()
    else:
        raise ValueError("Unsupported LLM output format")

    if not queries:
        raise ValueError("No queries found in LLM output")

    # dedupe & normalize whitespace
    seen = set()
    normed = []
    for q in queries:
        key = " ".join(q.split()).lower()
        if key not in seen:
            seen.add(key)
            normed.append(" ".join(q.split()))
    return normed, rationale


In [ ]:
print(queries)

In [ ]:
print(rationale)

In [221]:
query = input("Enter your question: ")
generated_query= generator_chain.invoke({"user_query":query})
queries, rationale = parse_llm_multiquery(generated_query)
print(queries)

Enter your question:  What is alzimers


["Definition of Alzheimer's disease (AD) and diagnostic criteria", 'What is Alzheimer’s disease? Clinical presentation and staging with MMSE, MoCA, CDR', 'Alzheimer’s disease biomarkers: amyloid β, p‑tau, CSF, PET, MRI hippocampal atrophy', 'Alzheimer’s disease differential diagnosis: MCI due to AD, vascular dementia, Lewy body dementia', 'Alzheimer’s disease management: cholinesterase inhibitors, memantine, lecanemab, lifestyle interventions']


In [ ]:
print(rationale)

In [ ]:

from typing import List, Dict, Any, Tuple
from chromadb import Knn, Search, K


In [ ]:
# ---- CONFIG ----
DENSE_KEY = "#embedding"          # default dense embedding field
SPARSE_KEY = "sparse_vector_key"  # your sparse vector metadata field
TOP_K_PER_BRANCH = 4              # k per branch per query
FINAL_TOP_K = 10                  # final top-k after merge
DENSE_WEIGHT = 0.6                # weight for dense score in fusion
SPARSE_WEIGHT = 0.4               # weight for sparse score in fusion


In [ ]:
print(all_results)

In [226]:
from chromadb import Knn, Rrf
from chromadb import Search
def all_results_retrieve(queries):
    all_results_dense = []
    all_results_sparse = []
    all_results=[]
    for q in queries:# Dense semantic embeddings
        dense_rank = Knn(
        query=q,  # Text query for dense embeddings
        key="#embedding",          # Default embedding field
        return_rank=True,
        )
    
        # Build and execute search
        dense_search = (Search()
          .rank(dense_rank)
          .select(K.DOCUMENT, K.SCORE, K.METADATA)
          .limit(5)
        )
      
        dense_results = collection.search(dense_search)
        all_results_dense.append(dense_results)
        #print(dense_results)
        
        sparse_rank = Knn(
          query=q,  # Text query for sparse embeddings
          key="sparse_vector_key",  # Metadata field for sparse vectors
          return_rank=True,
          limit=5                 # Only the 5 nearest documents get scored (default limit 16)
        )
        
        # Step 2: Build search
        sparse_search = (Search()
          .rank(sparse_rank)
          .select(K.DOCUMENT, K.SCORE)
          .limit(5)
        )
        
        # Step 3: Execute search
        sparse_results = collection.search(sparse_search)
        all_results_sparse.append(sparse_search)
        # Combine with RRF
        hybrid_rank = Rrf(
          ranks=[dense_rank, sparse_rank],
          weights=[0.7, 0.3],  # 70% semantic, 30% keyword
          k=60                 # smoothing parameter - higher values reduce emphasis on top ranks
        )
            # Use in search
        hybrid_search = (Search()
          .rank(hybrid_rank)
          .limit(3)
          .select(K.DOCUMENT, K.SCORE, K.METADATA)
        )
        
        hybrid_results = collection.search(hybrid_search)
        all_results.append(hybrid_results)
    return all_results
print(all_results)

[{'ids': [['a9a8e276-16aa-4459-924c-fb1c3a634250', '04173e54-a0df-483d-9e28-650ec6d8d7a4']], 'documents': [["Criteria\nThere are three sets of criteria for the clinical diagnoses of the spectrum of Alzheimer's disease: the 2013 fifth edition of the Diagnostic and Statistical Manual of Mental Disorders (DSM-5); the National Institute on Aging-Alzheimer's Association (NIA-AA) definition as revised in 2011; and the International Working Group criteria as revised in 2010.", 'Diagnosis\nPET scan of the brain of a person with Alzheimer\'s disease showing a loss of function in the temporal lobe\nAlzheimer\'s disease (AD) can only be definitively diagnosed with autopsy findings; in the absence of autopsy, clinical diagnoses of AD are "possible" or "probable", based on other findings.']], 'embeddings': [None], 'metadatas': [[{'source': 'data\\medical\\alzheimers_3.txt', 'sparse_vector_key': SparseVector(indices=[18314387, 22917276, 53716800, 88743589, 189262593, 240962094, 267514428, 294778681,

In [166]:
! pip install transformers torch --upgrade


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [168]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [169]:
_tok = AutoTokenizer.from_pretrained("BAAI/bge-reranker-base")
_mdl = AutoModelForSequenceClassification.from_pretrained("BAAI/bge-reranker-base")
_mdl.eval()

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=1, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Li

In [171]:

def _to_doc(x: dict):
    # Adjust if your key names differ (K.DOCUMENT usually maps to "document")
    return x.get("document") or x.get("doc") or x.get("text") or x.get("value")


In [172]:

def _to_id(x: dict):
    _id = x.get("id") or x.get("metadata", {}).get("id") or x.get("meta", {}).get("id")
    if _id is not None:
        return str(_id)
    d = _to_doc(x) or ""
    return str(hash(d))


In [173]:

def _minmax(xs):
    if not xs: return xs
    mn, mx = min(xs), max(xs)
    if mx == mn: return [0.5] * len(xs)
    return [(x - mn) / (mx - mn) for x in xs]


In [179]:

def _to_doc(x: dict):
    # Adjust if your key names differ (K.DOCUMENT usually maps to "document")
    return x.get("document") or x.get("doc") or x.get("text") or x.get("value")

def _to_id(x: dict):
    _id = x.get("id") or x.get("metadata", {}).get("id") or x.get("meta", {}).get("id")
    if _id is not None:
        return str(_id)
    d = _to_doc(x) or ""
    return str(hash(d))

def _minmax(xs):
    if not xs: return xs
    mn, mx = min(xs), max(xs)
    if mx == mn: return [0.5] * len(xs)
    return [(x - mn) / (mx - mn) for x in xs]

def _rrf_aggregate(ranks, k: int = 60) -> float:
    # sum(1 / (k + rank_i)) ; rank is 0-based here
    return sum(1.0 / (k + int(r)) for r in ranks)

def _invert_if_distance(score: float, is_distance: bool) -> float:
    if score is None: return 0.0
    if not is_distance: return float(score)
    return 1.0 / (1.0 + float(score))  # simple invert; adapt if you know metric

@torch.no_grad()
def rerank_multiquery_pool(
    original_query: str,
    all_results: list,       # list of hybrid result lists (5 lists of 5 hits) or list of {"results":[...]}
    top_k: int = 5,
    mix_with_orig: bool = True,
    alpha: float = 0.25,     # 0.2–0.3 usually good
    score_field: str = "score",
    score_is_distance: bool = False,
    max_length: int = 512,
    device: str | None = None
) -> list[dict]:
    """
    Late-stage re-ranking for multi-query hybrid RAG:
      - Flatten 5×5 hits -> pool
      - Dedupe by id
      - Aggregate retrieval signal (max score + RRF over per-subquery ranks)
      - Cross-encoder rerank with original user query (BAAI/bge-reranker-base)
      - Optional blend of aggregated score and reranker score
      - Return top_k

    Returns items with: id, document, metadata, provenance, raw_reranker, rerank_score, agg_score
    """

    # 1) Normalize input and keep provenance (subquery index + rank)
    pool = []
    for sub_idx, sub in enumerate(all_results):
        items = (sub.get("results") if isinstance(sub, dict) and "results" in sub
                 else getattr(sub, "results", None) or sub)
        if not items:
            continue
        for rank_idx, it in enumerate(items):
            it = dict(it)  # shallow copy
            it["_subquery_index"] = sub_idx
            it["_rank_in_subquery"] = rank_idx
            pool.append(it)

    if not pool:
        return []

    # 2) Group by id (dedupe) and aggregate original scores/ranks
    by_id = {}
    for it in pool:
        _id = _to_id(it)
        doc = _to_doc(it)
        if not doc:
            continue
        sim = _invert_if_distance(it.get(score_field), score_is_distance)
        rnk = int(it.get("_rank_in_subquery", 0))
        entry = by_id.setdefault(_id, {
            "id": _id,
            "document": doc,
            "metadata": it.get("metadata") or it.get("meta") or {},
            "provenance": [],
            "scores": [],
            "ranks": []
        })
        entry["provenance"].append({
            "subquery": it.get("_subquery_index"),
            "rank": rnk,
            "score": sim
        })
        entry["scores"].append(sim)
        entry["ranks"].append(rnk)

    # 3) Aggregate original retrieval signal per id
    pooled = []
    for _id, row in by_id.items():
        if not row["document"]:
            continue
        max_score = max(row["scores"]) if row["scores"] else 0.0
        rrf_score = _rrf_aggregate(row["ranks"], k=60) if row["ranks"] else 0.0
        agg = max_score + rrf_score
        pooled.append({
            "id": _id,
            "document": row["document"],
            "metadata": row["metadata"],
            "provenance": row["provenance"],
            "agg_score": agg
        })

    if not pooled:
        return []

    # 4) Cross-encoder rerank with original user query
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    _mdl.to(device)

    docs = [c["document"] for c in pooled]
    enc = _tok([original_query] * len(docs), docs,
                padding=True, truncation=True, max_length=max_length,
                return_tensors="pt").to(device)
    logits = _mdl(**enc).logits.view(-1)      # higher = better
    rr_scores = logits.detach().cpu().tolist()

    # 5) Optional blend with aggregated orig score
    if mix_with_orig:
        agg = [c["agg_score"] for c in pooled]
        final = [alpha * o + (1 - alpha) * r for o, r in zip(_minmax(agg), _minmax(rr_scores))]
    else:
        final = rr_scores

    for c, r_raw, r_fin in zip(pooled, rr_scores, final):
        c["raw_reranker"] = float(r_raw)
        c["rerank_score"] = float(r_fin)

    # 6) Sort, take top_k
    return sorted(pooled, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

In [180]:
print(query)

what is alzimers


In [178]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [183]:
from typing import Any, Iterable

# Assumes you've already created:
# _tok = AutoTokenizer.from_pretrained("BAAI/bge-reranker-base")
# _mdl = AutoModelForSequenceClassification.from_pretrained("BAAI/bge-reranker-base"); _mdl.eval()

# --------------------- shape helpers ---------------------
def _to_doc_from_any(x: Any):
    """Extract text content from many possible shapes."""
    if x is None:
        return None
    if isinstance(x, dict):
        return x.get("document") or x.get("doc") or x.get("text") or x.get("value")
    if hasattr(x, "model_dump"):  # pydantic v2
        d = x.model_dump()
        return d.get("document") or d.get("doc") or d.get("text") or d.get("value")
    if hasattr(x, "dict"):  # pydantic v1
        d = x.dict()
        return d.get("document") or d.get("doc") or d.get("text") or d.get("value")
    if hasattr(x, "_asdict"):  # namedtuple
        d = x._asdict()
        return d.get("document") or d.get("doc") or d.get("text") or d.get("value")
    if isinstance(x, str):
        return x
    if isinstance(x, (list, tuple)) and x:
        # Heuristic: (document, score, metadata, id)
        return x[0] if isinstance(x[0], str) else None
    return None

def _to_id_from_any(x: Any, doc_text: str | None):
    """Prefer explicit id; fallback to metadata.id; else hash of doc."""
    if isinstance(x, dict):
        _id = x.get("id") or x.get("metadata", {}).get("id") or x.get("meta", {}).get("id")
        if _id is not None:
            return str(_id)
    if hasattr(x, "model_dump"):
        d = x.model_dump()
        _id = d.get("id") or d.get("metadata", {}).get("id") or d.get("meta", {}).get("id")
        if _id is not None:
            return str(_id)
    if hasattr(x, "dict"):
        d = x.dict()
        _id = d.get("id") or d.get("metadata", {}).get("id") or d.get("meta", {}).get("id")
        if _id is not None:
            return str(_id)
    if hasattr(x, "_asdict"):
        d = x._asdict()
        _id = d.get("id") or d.get("metadata", {}).get("id") or d.get("meta", {}).get("id")
        if _id is not None:
            return str(_id)
    if isinstance(x, (list, tuple)) and len(x) >= 4:
        return str(x[3])
    return str(hash(doc_text or ""))

def _to_meta_from_any(x: Any):
    if isinstance(x, dict):
        return x.get("metadata") or x.get("meta") or {}
    if hasattr(x, "model_dump"):
        d = x.model_dump()
        return d.get("metadata") or d.get("meta") or {}
    if hasattr(x, "dict"):
        d = x.dict()
        return d.get("metadata") or d.get("meta") or {}
    if hasattr(x, "_asdict"):
        d = x._asdict()
        return d.get("metadata") or d.get("meta") or {}
    if isinstance(x, (list, tuple)) and len(x) >= 3 and isinstance(x[2], dict):
        return x[2]
    return {}

def _to_score_from_any(x: Any, score_field: str = "score"):
    if isinstance(x, dict):
        if score_field in x: return x[score_field]
        for k in ("score", "similarity", "distance", "dist", "relevance"):
            if k in x: return x[k]
    if hasattr(x, "model_dump"):
        d = x.model_dump()
        if score_field in d: return d[score_field]
        for k in ("score", "similarity", "distance", "dist", "relevance"):
            if k in d: return d[k]
    if hasattr(x, "dict"):
        d = x.dict()
        if score_field in d: return d[score_field]
        for k in ("score", "similarity", "distance", "dist", "relevance"):
            if k in d: return d[k]
    if hasattr(x, "_asdict"):
        d = x._asdict()
        if score_field in d: return d[score_field]
        for k in ("score", "similarity", "distance", "dist", "relevance"):
            if k in d: return d[k]
    if isinstance(x, (list, tuple)) and len(x) >= 2 and isinstance(x[1], (int, float)):
        return x[1]
    return None

def _flatten_items_with_provenance(all_results: list) -> list[dict]:
    """
    Accepts per-subquery results (each can be:
      - list[dict|str|tuple]
      - {"results": [...]}
      - object with .results
      - Chroma-like dict with 'documents'/'ids'/'metadatas'/'distances'|'scores'
    Returns list of normalized dict hits with provenance fields.
    """
    pool = []
    for sub_idx, sub in enumerate(all_results):
        # Chroma query-like dict (ids, documents, metadatas, distances/scores)
        if isinstance(sub, dict) and "documents" in sub:
            docs = sub["documents"]
            ids = sub.get("ids")
            metas = sub.get("metadatas")
            dists = sub.get("distances") or sub.get("scores")
            # docs may be list-of-lists or flat list—handle both
            if docs and isinstance(docs[0], list):
                docs_list = docs[0]
                ids_list = ids[0] if ids and isinstance(ids[0], list) else ids
                metas_list = metas[0] if metas and isinstance(metas[0], list) else metas
                dists_list = dists[0] if dists and isinstance(dists[0], list) else dists
            else:
                docs_list, ids_list, metas_list, dists_list = docs, ids, metas, dists

            for rank_idx, _doc in enumerate(docs_list or []):
                _id = (ids_list[rank_idx] if ids_list else None)
                _meta = (metas_list[rank_idx] if metas_list else {})
                _score = (dists_list[rank_idx] if dists_list else None)
                pool.append({
                    "id": str(_id) if _id is not None else str(hash(_doc)),
                    "document": _doc,
                    "metadata": _meta or {},
                    "score": _score,
                    "_subquery_index": sub_idx,
                    "_rank_in_subquery": rank_idx,
                })
            continue

        # Generic list/object with .results
        items = (sub.get("results") if isinstance(sub, dict) and "results" in sub
                 else getattr(sub, "results", None) or sub)

        # Ensure iterable, avoid iterating strings/bytes
        if not isinstance(items, Iterable) or isinstance(items, (str, bytes)):
            items = [items]

        for rank_idx, raw in enumerate(items):
            doc = _to_doc_from_any(raw)
            if not doc:
                try:
                    doc = str(raw)
                except Exception:
                    continue
            pool.append({
                "id": _to_id_from_any(raw, doc),
                "document": doc,
                "metadata": _to_meta_from_any(raw),
                "score": _to_score_from_any(raw),
                "_subquery_index": sub_idx,
                "_rank_in_subquery": rank_idx,
            })
    return pool

def _minmax(xs):
    if not xs: return xs
    mn, mx = min(xs), max(xs)
    if mx == mn: return [0.5] * len(xs)
    return [(x - mn) / (mx - mn) for x in xs]

def _rrf_aggregate(ranks, k: int = 60) -> float:
    return sum(1.0 / (k + int(r)) for r in ranks)

def _invert_if_distance(score: float | None, is_distance: bool) -> float:
    if score is None: return 0.0
    return (1.0 / (1.0 + float(score))) if is_distance else float(score)

# --------------------- main function ---------------------
@torch.no_grad()
def rerank_multiquery_pool(
    original_query: str,
    all_results: list,       # 5 subquery results, each ~5 hits
    top_k: int = 5,
    mix_with_orig: bool = True,
    alpha: float = 0.25,
    score_field: str = "score",
    score_is_distance: bool = False,
    max_length: int = 512,
    device: str | None = None
) -> list[dict]:
    # 1) Flatten & normalize
    raw_pool = _flatten_items_with_provenance(all_results)
    if not raw_pool:
        return []

    # 2) Group by id (dedupe) & aggregate retrieval signal
    by_id = {}
    for it in raw_pool:
        _id = it["id"]
        sim = _invert_if_distance(it.get(score_field), score_is_distance)
        rnk = int(it.get("_rank_in_subquery", 0))
        row = by_id.setdefault(_id, {
            "id": _id,
            "document": it["document"],
            "metadata": it.get("metadata", {}),
            "provenance": [],
            "scores": [],
            "ranks": [],
        })
        row["provenance"].append({
            "subquery": it.get("_subquery_index"),
            "rank": rnk,
            "score": sim
        })
        row["scores"].append(sim)
        row["ranks"].append(rnk)

    pooled = []
    for _id, row in by_id.items():
        if not row["document"]:
            continue
        max_s = max(row["scores"]) if row["scores"] else 0.0
        rrf_s = _rrf_aggregate(row["ranks"], k=60) if row["ranks"] else 0.0
        agg = max_s + rrf_s
        pooled.append({
            "id": _id,
            "document": row["document"],
            "metadata": row["metadata"],
            "provenance": row["provenance"],
            "agg_score": agg
        })

    if not pooled:
        return []

    # 3) Cross-encoder rerank with original user query
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    _mdl.to(device)

    docs = [c["document"] for c in pooled]
    enc = _tok([original_query] * len(docs), docs,
               padding=True, truncation=True, max_length=max_length,
               return_tensors="pt").to(device)
    logits = _mdl(**enc).logits.view(-1)
    rr_scores = logits.detach().cpu().tolist()

    # 4) Optional blend with aggregated score
    if mix_with_orig:
        agg = [c["agg_score"] for c in pooled]
        final = [alpha * o + (1 - alpha) * r for o, r in zip(_minmax(agg), _minmax(rr_scores))]
    else:
        final = rr_scores

    for c, r_raw, r_fin in zip(pooled, rr_scores, final):
        c["raw_reranker"] = float(r_raw)
        c["rerank_score"] = float(r_fin)

    return sorted(pooled, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

In [184]:
print(query)

what is alzimers


In [185]:
print(all_results)

[{'ids': [['a9a8e276-16aa-4459-924c-fb1c3a634250', '04173e54-a0df-483d-9e28-650ec6d8d7a4']], 'documents': [["Criteria\nThere are three sets of criteria for the clinical diagnoses of the spectrum of Alzheimer's disease: the 2013 fifth edition of the Diagnostic and Statistical Manual of Mental Disorders (DSM-5); the National Institute on Aging-Alzheimer's Association (NIA-AA) definition as revised in 2011; and the International Working Group criteria as revised in 2010.", 'Diagnosis\nPET scan of the brain of a person with Alzheimer\'s disease showing a loss of function in the temporal lobe\nAlzheimer\'s disease (AD) can only be definitively diagnosed with autopsy findings; in the absence of autopsy, clinical diagnoses of AD are "possible" or "probable", based on other findings.']], 'embeddings': [None], 'metadatas': [[{'source': 'data\\medical\\alzheimers_3.txt', 'sparse_vector_key': SparseVector(indices=[18314387, 22917276, 53716800, 88743589, 189262593, 240962094, 267514428, 294778681,

In [186]:
final_top5 = rerank_multiquery_pool(
    original_query=query,     # the original single user query
    all_results=all_results,  # 5 subquery hybrid results (each ~5 hits)
    top_k=5,
    mix_with_orig=True,
    alpha=0.25,
    score_field="score",
    score_is_distance=False
)

In [187]:
print()

[{'id': '0122baca-1f7c-4a71-bb94-b9af5ad5d0de', 'document': "The third stage is divided into probable and possible AD dementia. In probable AD dementia there is steady impairment of cognition over time and a memory-related or non-memory-related cognitive dysfunction. In possible AD dementia, another causal disease such as cerebrovascular disease is present.\n\nTechniques\n\nCognitive tests such as the miniâ€“mental state examination (MMSE) can help in the diagnosis of Alzheimer's disease.", 'metadata': {'sparse_vector_key': SparseVector(indices=[18314387, 262179772, 359483635, 367477241, 589901640, 722829366, 829393320, 1075238684, 1105706408, 1157923234, 1167338989, 1221514274, 1271291307, 1338150097, 1391639301, 1402904868, 1615913549, 1627331861, 1663403958, 1754526443, 1758470515, 1875637301, 1902821451, 1903244910, 1937451471, 1942200790, 1952394616, 2058513491, 2119175981, 2129237020], values=[1.5196978, 1.5196978, 1.5196978, 1.797638, 1.5196978, 1.5196978, 1.5196978, 1.797638, 1

In [188]:
from langchain_core.prompts import ChatPromptTemplate

In [256]:
MULTI_CONTEXT_PROMPT ="""
You are a careful, precise assistant. Use ONLY the provided context to answer. 

Answer the question strictly using the information in the provided context.
Consider all Question and Sub Questions as one and provide answer only once (mandatory)
#########
{context}
#########
Question: {question}
#########
S
From the user query subqueries also derived by the system for you, Analyse the subquery also, but user question has the priority.
Sub Questions:{sub_query}
Instructions:
- Understand the context and the question semantically, not just by keywords.
- Give a direct and precise answer strictly based on the provided context.
- Do NOT justify your answer.
- Do NOT add information that is not present in the context.
- Do NOT mention the context explicitly.
- If the question cannot be answered from the context, or if there is no meaningful semantic similarity between the question and the context, respond exactly with:
  "This topic is not within my expertise. You may need to ask someone else."
########
"""

In [257]:
answer_chat_model = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=None,           
    reasoning_format="hidden", 
    max_retries=2,
    model_kwargs={      
        "top_p": 1.0,
      },
)

In [258]:
answer_prompt_template = ChatPromptTemplate(
    messages=[
        MULTI_CONTEXT_PROMPT
    ]
)

In [259]:
answer_parser = StrOutputParser()

In [208]:
from langchain_core.runnables import RunnablePassthrough

In [233]:

def format_docs(docs):
    return "\n\n".join(
        (d.page_content if hasattr(d, "page_content") else d.get("document", "")) 
        for d in docs
        if (hasattr(d, "page_content") and d.page_content) or (isinstance(d, dict) and d.get("document"))
    )


In [194]:
print(queries)

['Alzheimer’s disease definition and diagnostic criteria', 'Differential diagnosis of Alzheimer’s disease versus MCI, mild cognitive impairment, and other dementias', 'Biomarker panel for Alzheimer’s: amyloid PET, CSF Aβ42, p‑tau, and hippocampal atrophy on MRI', 'Guideline recommendations for Alzheimer’s treatment: cholinesterase inhibitors, memantine, and anti‑amyloid therapies lecanemab and donanemab', 'Staging Alzheimer’s disease with MMSE, MoCA, CDR and assessment of ADLs/IADLs']


In [ ]:
def format_docs(items):
    return "\n\n".join(item["document"] for item in items if "document" in item)

In [239]:

def extract_all_documents(results):
    all_docs = []
    for item in results:
        docs = item.get("documents", [])
        # each "documents" entry is a list of lists → flatten
        for doc_group in docs:
            for doc in doc_group:
                all_docs.append(doc)
    return "\n\n".join(all_docs)


In [223]:
query = input("Enter your question: ")
multi_query = query_chain.invoke({"user_query":query})

Enter your question:  What is alzimers


In [224]:
queries, rationale = parse_llm_multiquery(multi_query)
print(queries)

["Alzheimer's disease: definition, symptoms, and diagnosis", 'Diagnostic criteria for Alzheimer’s disease and prodromal AD', 'Alzheimer’s disease overview: clinical features, biomarkers, and differential diagnosis', 'Stages of Alzheimer’s disease: MCI due to AD, early dementia, and functional decline', 'AT(N) framework and APOE ε4 in Alzheimer’s disease: biomarker and risk assessment']


In [ ]:
print(all_results)

In [227]:
all_results = all_results_retrieve(queries)

In [235]:
answer_chain = answer_prompt_template | answer_chat_model | answer_parser

In [242]:
answer = answer_chain.invoke({"question":queries,"sub_query":queries,"context":extract_all_documents(all_results)})

In [243]:
print(answer)

**Alzheimer’s disease (AD)**  
- **Definition** – A protein‑misfolding neurodegenerative disorder marked by amyloid‑β (Aβ) plaques and tau‑based neurofibrillary tangles that progressively impair cognition.  

**Key clinical features**  
- Progressive learning and memory loss; other deficits (language, executive function, perception, apraxia) may dominate in a minority.  
- Apathy and depression are common; apathy tends to persist throughout the disease.  
- Subtle cognitive difficulties can be detected up to ~8 years before formal diagnosis.  

**Diagnostic approach**  
- **Definitive diagnosis** requires autopsy.  
- **Clinical diagnosis** is “possible” or “probable” and relies on:  
  - Neuropsychological testing (identifies mild cognitive impairment, especially amnestic MCI, which has >90 % likelihood of AD).  
  - Biomarkers:  
    - Imaging of Aβ and tau deposition (PET shows temporal‑lobe dysfunction).  
    - CSF or blood measures of Aβ and tau (e.g., the FDA‑approved Lumipulse 

In [260]:

from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


In [261]:

def _parse_multi(multi_query_output):
    queries, rationale = parse_llm_multiquery(multi_query_output)
    return {"queries": queries, "rationale": rationale}


In [262]:

make_multi = (
    {"user_query": RunnablePassthrough()}      # accept a raw string and map to the expected input
    | query_chain                               # your LLM chain that produces multi_query text
    | RunnableLambda(_parse_multi)              # -> {"queries": [...], "rationale": "..."}
)


In [263]:

parallel_stage = {
    "queries": itemgetter("queries"),
    "rationale": itemgetter("rationale"),
    "all_results": itemgetter("queries") | RunnableLambda(all_results_retrieve),
}


In [264]:

to_answer_inputs = RunnableLambda(lambda x: {
    "question": x["queries"],                 # you passed queries in your original call
    "sub_query": x["queries"],                # same as above
    "context": extract_all_documents(x["all_results"])
})


In [265]:

# 5) Full pipeline
full_chain = make_multi | parallel_stage | to_answer_inputs | answer_chain


In [250]:
answer = await full_chain.ainvoke("Enter your question: ")

In [251]:
print(answer)

**Alzheimer’s disease diagnosis criteria, biomarkers, and imaging**  
- Clinical diagnosis is based on the presence of cognitive impairment without comorbidities.  
- Staging is divided into pre‑clinical, mild cognitive impairment (MCI), and Alzheimer’s dementia.  
- Biomarkers: early and robust changes in amyloid‑β (Aβ) are detected by imaging of protein deposits and by measuring brain‑derived substances in cerebrospinal fluid (CSF) and blood. Tau abnormalities (e.g., CSF p‑tau) are also used, especially for neuronal injury.  
- Imaging: FDG‑PET is not required but may be used when standard testing is unclear; it shows reduced activity in temporal and parietal regions. FDA‑approved PET tracers for Aβ are florbetapir, flutemetamol, florbetaben, and for tau is flortaucipir.  
- Definitive diagnosis requires autopsy; otherwise diagnoses are “possible” or “probable” based on clinical and biomarker findings.

**Efficacy of donepezil and rivastigmine in mild cognitive impairment due to Alzh

In [266]:
query = input("Enter your question: ")
answer = await full_chain.ainvoke(query)

Enter your question:  What is the most common cause of dementia?


In [267]:
print(answer)

**Most common cause of dementia**  
Alzheimer’s disease.

**Leading cause of dementia in adults**  
Alzheimer’s disease.

**Alzheimer’s disease – statistics and prevalence**  
- Accounts for about 60–70 % of all dementia cases.  
- Prevalence in the United States (2020):  
  - 5.3 % in the 60–74 age group  
  - 13.8 % in the 74–84 age group  
  - 34.6 % in those older than 85.

**Lead cause of neurocognitive disorder dementia (Alzheimer’s vs vascular dementia)**  
Alzheimer’s disease is the leading cause, with vascular dementia also common but less frequent.

**Common cause of dementia in older adults according to AAN 2023 guidelines**  
This topic is not within my expertise. You may need to ask someone else.
